In [1]:
from common.corpus import load_corpus
from common import llm

text = {d.doc_id: d.text for d in load_corpus()}

dataset = [
    {"qid": "e1", "frage": "Wie hoch ist der zulässige Dauerbetriebsdruck?",
     "kontext": text["p12-betriebsdruck"],
     "antwort": "Der zulässige Dauerbetriebsdruck beträgt 350 bar [p12-betriebsdruck].",
     "gold": 5},
    {"qid": "e2", "frage": "Wie hoch ist der zulässige Dauerbetriebsdruck?",
     "kontext": text["p12-betriebsdruck"],
     "antwort": "Im Dauerbetrieb sind 400 bar zulässig und empfohlen.",
     "gold": 1},
    {"qid": "e3", "frage": "Welches Hydrauliköl ist vorgeschrieben?",
     "kontext": text["p12-hydraulikoel"],
     "antwort": "Vorgeschrieben ist HLP 46 nach DIN 51524 Teil 2 [p12-hydraulikoel].",
     "gold": 5},
    {"qid": "e4", "frage": "Welches Öl und in welchem Intervall wird gewechselt?",
     "kontext": text["p12-hydraulikoel"],
     "antwort": "HLP 46, Wechsel alle 1000 Betriebsstunden.",
     "gold": 2},
    {"qid": "e5", "frage": "Wann ist der erste Ölwechsel fällig?",
     "kontext": text["p12-wartungsintervalle"],
     "antwort": "Das erste Öl wird nach 50 Betriebsstunden gewechselt [p12-wartungsintervalle].",
     "gold": 5},
    {"qid": "e6", "frage": "Wie lange gilt die Werksgarantie?",
     "kontext": text["p12-datenblatt"],
     "antwort": "Die Werksgarantie beträgt 24 Monate.",
     "gold": 2},
]
print(f"{len(dataset)} Eval-Einträge, Gold-Verteilung:",
      {g: sum(1 for d in dataset if d['gold'] == g) for g in sorted({d['gold'] for d in dataset})})


6 Eval-Einträge, Gold-Verteilung: {1: 1, 2: 2, 5: 3}


In [2]:
import json
import re

JUDGE_SYSTEM = """Sie sind ein präziser Evaluator für RAG-Antworten. Bewerten Sie
die Faithfulness (Treue zum Kontext) auf einer Skala von 1 bis 5.

Rubrik:
5 = jede Aussage wird vom Kontext gestützt.
4 = im Kern gestützt, kleine unbelegte Nebenaussage.
3 = teils gestützt, teils unbelegt.
2 = wesentliche Aussage unbelegt oder nur schwach gestützt.
1 = widerspricht dem Kontext oder ist frei erfunden.

Bewerten Sie ausschließlich anhand des Kontexts, nicht anhand Ihres Weltwissens.
Antworten Sie als JSON: {"score": <int 1-5>, "begruendung": "<ein Satz>"}."""


def judge(frage: str, kontext: str, antwort: str) -> dict:
    prompt = (f"Frage: {frage}\n\nKontext:\n{kontext}\n\nZu bewertende Antwort:\n"
              f"{antwort}\n\nIhr Urteil als JSON:")
    roh = llm.complete(prompt, system=JUDGE_SYSTEM, temperature=0.0, max_tokens=512)
    m = re.search(r"\{.*\}", roh, re.DOTALL)
    try:
        obj = json.loads(m.group(0)) if m else {}
        return {"score": int(obj.get("score", 0)), "begruendung": str(obj.get("begruendung", ""))}
    except (json.JSONDecodeError, ValueError):
        return {"score": 0, "begruendung": f"PARSE-FEHLER: {roh[:80]}"}


In [3]:
for d in dataset:
    r = judge(d["frage"], d["kontext"], d["antwort"])
    d["judge"] = r["score"]
    print(f"{d['qid']}  gold={d['gold']}  judge={r['score']}  {r['begruendung'][:70]}")

e1  gold=5  judge=5  Die Aussage wird vollständig vom Kontext gestützt, da der Dauerbetrieb
e2  gold=1  judge=1  Die Aussage widerspricht dem Kontext, da der Dauerbetriebsdruck mit 35
e3  gold=5  judge=5  Die Aussage wird vollständig vom Kontext gestützt, da HLP 46 nach DIN 
e4  gold=2  judge=2  Die Aussage über den Wechselintervall von 1000 Betriebsstunden ist wes
e5  gold=5  judge=5  Die Aussage wird durch den Kontext vollständig gestützt, da dort expli
e6  gold=2  judge=1  Die Antwort zur Werksgarantie ist nicht im Kontext enthalten und somit


In [4]:
from sklearn.metrics import cohen_kappa_score

gold = [d["gold"] for d in dataset]
jud = [d["judge"] for d in dataset]
mae = sum(abs(a - b) for a, b in zip(gold, jud)) / len(gold)
kappa = cohen_kappa_score(gold, jud, weights="quadratic")
    
print(f"Mittlere absolute Abweichung: {mae:.2f} Notenpunkte")
kappa

Mittlere absolute Abweichung: 0.17 Notenpunkte


0.88